In [5]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib.patches import Rectangle
from datetime import timedelta
import logging
import mysql

In [6]:
# Prepare the DataFrame for the investment window analysis by ensuring proper data types and sorting.
def prepare_data(df, asset_class=None, start=pd.Timestamp.now(), end=pd.DateOffset(months=3)):
    """
    Prepare and clean the DataFrame for analysis.
    Args:
        df: DataFrame with 'TransactionDate', 'TransactionClass', and 'Available' columns
        asset_class: String to filter TransactionClass column (e.g., 'US Agencies')
        start_date_parameter: Starting date for analysis, defaults to today
        end_date_parameter: Ending date for analysis, defaults to 3 months from today
    Returns:
        DataFrame: Cleaned and sorted DataFrame
    """
    # Define start and end dates
    start_date = pd.Timestamp(start).normalize()
    end_date = pd.Timestamp(end).normalize()
    
    # 1. Filter the data within the date range
    data = df[(df['TransactionDate'] >= start_date) & (df['TransactionDate'] <= end_date)].copy()
    
    # 2. Filter by asset class if specified
    if asset_class is not None:
        # Check if asset_class exists in TransactionClass column
        if asset_class not in df['TransactionClass'].unique():
            raise ValueError(f"Asset class '{asset_class}' not found in data")
        data = data[data['TransactionClass'] == asset_class].copy()
        if data.empty:
            raise ValueError(f"No data found for asset class: {asset_class}")

    # Set the TransactionDate to datetime
    data['TransactionDate'] = pd.to_datetime(data['TransactionDate'])

    # Sort
    data = data.sort_values('TransactionDate').reset_index(drop=True)
    data['TransactionClass'] = asset_class if asset_class else 'Not Specified'
    return data[['TransactionDate', 'Available','TransactionClass']]

In [7]:
# MAIN (DEBUG)
#     analyze_balance_data() for running the asset class balances processing

# In prod we'll be looping over each asset class and process the investment windows one by one
asset_classes = ['Certificate of Deposit', 'Mutual Fund', 'Commercial Paper', 'Money Market', 'US Treasuries', 'US Agencies']
asset_index = 4
print("Asset class:", asset_classes[asset_index])
# Load data
running_balances = pd.read_pickle('running_balances.pkl')
data = prepare_data(running_balances, asset_classes[asset_index], '2025-09-04', '2025-10-31')

#Display
# print (asset_classes[asset_index],":\n")
# print(data)
with pd.option_context('display.max_columns', None,
                       'display.max_rows', None,
                       'display.width', None,
                       'display.expand_frame_repr', False):
    pd.options.display.float_format = '${:,.2f}'.format
    # display(data.head(15))

Asset class: US Treasuries


In [8]:
def find_exact_intervals(df, threshold):
    """Find intervals where the balance is exactly a specific amount."""
    start_dates = []
    end_dates = []
    amounts = []
    
    current_amount = df.iloc[0]['Available']
    current_start = df.iloc[0]['TransactionDate']
    prev_date = df.iloc[0]['TransactionDate']
    
    for i in range(1, len(df)):
        current_row = df.iloc[i]
        
        if current_row['Available'] != current_amount or i == len(df) - 1:
            if current_amount >= threshold:
                start_dates.append(current_start)
                end_dates.append(prev_date)
                amounts.append(current_amount)
            
            current_amount = current_row['Available']
            current_start = current_row['TransactionDate']
        
        prev_date = current_row['TransactionDate']
    
    return start_dates, end_dates, amounts

In [9]:
def find_consecutive_periods_for_amount(df, target_amount):
    """Find all consecutive periods where at least target_amount is available."""
    consecutive_periods = []
    current_start = None
    
    for i in range(len(df)):
        current_available = df.iloc[i]['Available']
        current_date = df.iloc[i]['TransactionDate']
        
        if current_available >= target_amount:
            if current_start is None:
                current_start = current_date
        else:
            if current_start is not None:
                consecutive_periods.append((current_start, df.iloc[i-1]['TransactionDate']))
                current_start = None
    
    # Handle case where the last period extends to the end
    if current_start is not None:
        consecutive_periods.append((current_start, df.iloc[-1]['TransactionDate']))
    
    return consecutive_periods

In [10]:
def get_longest_period(consecutive_periods):
    """Get the longest period from a list of consecutive periods."""
    if not consecutive_periods:
        return None
    
    return max(consecutive_periods, 
               key=lambda x: (pd.to_datetime(x[1]) - pd.to_datetime(x[0])).days)


In [11]:
def is_duplicate_interval(start_date, end_date, amount, existing_starts, existing_ends, existing_amounts, existing_types):
    """Check if an interval is already captured in existing exact intervals."""
    for i in range(len(existing_starts)):
        if (existing_types[i] == 'exact' and 
            existing_amounts[i] == amount and
            existing_starts[i] == start_date and 
            existing_ends[i] == end_date):
            return True
    return False

In [12]:
def find_minimum_available_intervals(df, threshold, existing_starts, existing_ends, existing_amounts, existing_types):
    """Find the longest available intervals for each unique amount level."""
    start_dates = []
    end_dates = []
    amounts = []
    
    unique_amounts = sorted(df['Available'].unique())
    
    for target_amount in unique_amounts:
        if target_amount < threshold:
            continue
        
        consecutive_periods = find_consecutive_periods_for_amount(df, target_amount)
        longest_period = get_longest_period(consecutive_periods)
        
        if longest_period:
            longest_start, longest_end = longest_period
            
            if not is_duplicate_interval(longest_start, longest_end, target_amount,
                                       existing_starts, existing_ends, existing_amounts, existing_types):
                start_dates.append(longest_start)
                end_dates.append(longest_end)
                amounts.append(target_amount)
    
    return start_dates, end_dates, amounts

In [13]:
def calculate_interval_metrics(intervals_df):
    """Calculate duration and difference metrics for intervals."""
    intervals_df['Duration'] = (intervals_df['EndDate'] - intervals_df['StartDate']).dt.days + 1
    intervals_df['Difference'] = intervals_df['Amount'].diff()
    intervals_df.loc[0, 'Difference'] = intervals_df.loc[0, 'Amount']
    return intervals_df


In [14]:
def find_balance_intervals(df, threshold=1_000_000):
    """
    Find balance intervals including both exact amounts and minimum available periods.
    
    Returns intervals of two types:
    - 'exact': Periods where balance is exactly a specific amount
    - 'minimum_available': Longest periods where at least a specific amount is available
    """
    # Find exact intervals
    exact_starts, exact_ends, exact_amounts = find_exact_intervals(df, threshold)
    exact_types = ['exact'] * len(exact_starts)
    
    # Find minimum available intervals
    min_starts, min_ends, min_amounts = find_minimum_available_intervals(
        df, threshold, exact_starts, exact_ends, exact_amounts, exact_types
    )
    min_types = ['minimum_available'] * len(min_starts)
    
    # Combine all intervals
    all_starts = exact_starts + min_starts
    all_ends = exact_ends + min_ends
    all_amounts = exact_amounts + min_amounts
    all_types = exact_types + min_types
    
    # Create and sort result dataframe
    intervals_df = pd.DataFrame({
        'StartDate': all_starts,
        'EndDate': all_ends,
        'Amount': all_amounts,
        'IntervalType': all_types
    })
    
    intervals_df = intervals_df.sort_values('StartDate').reset_index(drop=True)
    
    # Calculate metrics
    intervals_df = calculate_interval_metrics(intervals_df)
    
    return intervals_df

In [15]:
# Run the function on the data
intervals = find_balance_intervals(data)
display(intervals)

,StartDate,EndDate,Amount,IntervalType,Duration,Difference
0,2025-09-04,2025-09-04,"$18,780,124.55",exact,1,"$18,780,124.55"
1,2025-09-04,2025-10-09,"$17,860,124.55",minimum_available,36,"$-920,000.00"
2,2025-09-05,2025-09-07,"$17,860,124.55",exact,3,$0.00
3,2025-09-08,2025-10-09,"$27,320,124.55",minimum_available,32,"$9,460,000.00"
4,2025-09-08,2025-10-09,"$18,780,124.55",minimum_available,32,"$-8,540,000.00"
5,2025-09-08,2025-10-02,"$42,086,124.55",minimum_available,25,"$23,306,000.00"
6,2025-09-08,2025-10-02,"$42,258,124.55",minimum_available,25,"$172,000.00"
7,2025-09-08,2025-10-05,"$39,486,124.55",minimum_available,28,"$-2,772,000.00"
8,2025-09-08,2025-10-05,"$39,886,124.55",minimum_available,28,"$400,000.00"
9,2025-09-08,2025-09-08,"$84,634,124.55",exact,1,"$44,748,000.00"
